# 1. Driver check


In [ ]:
import spcm
from spcm import units

with spcm.Card(card_type=spcm.SPCM_TYPE_AO, verbose=True) as card:
    print(f"Serial number:    {card.sn()}")
    print(f"Function type:    {card.function_type()}")
    print(f"Max sample value: {card.max_sample_value()}")

    clock = spcm.Clock(card)
    max_rate = clock.sample_rate(max=True, return_unit=units.MHz)
    print(f"Max sample rate:  {max_rate}")

print("Driver check OK -- card opened, queried, and closed without errors.")

# 2. Spectrum analyzer connection (Siglent SVA1015X)


In [ ]:
import pyvisa

rm = pyvisa.ResourceManager()
print(rm.list_resources())  # run once to find the exact VISA resource string below

('ASRL1::INSTR', 'ASRL2::INSTR', 'ASRL3::INSTR', 'ASRL4::INSTR', 'ASRL5::INSTR', 'USB0::0xF4EC::0x1301::SVA1XA1Q800517::INSTR')


In [ ]:
# SVA_RESOURCE = "TCPIP::192.168.1.100::inst0::INSTR"  # LAN (VXI-11) -- replace with the analyzer's IP
SVA_RESOURCE = "USB0::0xF4EC::0x1301::SVA1XA1Q800517::INSTR"  # USB-TMC -- get the exact string from rm.list_resources()

sva = rm.open_resource(SVA_RESOURCE)
sva.timeout = 5000  # ms
sva.read_termination = "\n"
sva.write_termination = "\n"

idn = sva.query("*IDN?").strip()
print(f"Connected to: {idn}")
assert "SVA1015X" in idn, f"Unexpected instrument reply: {idn!r}"

Connected to: Siglent Technologies,SVA1015X,SVA1XA1Q800517,3.2.2.6.0R10


In [ ]:
def set_span(center_hz, span_hz):
    """Center the analyzer's sweep on center_hz with the given span (Hz)."""
    sva.write(f":SENSe:FREQuency:CENTer {center_hz}")
    sva.write(f":SENSe:FREQuency:SPAN {span_hz}")


def read_peak():
    """Move marker 1 to the highest peak in the current sweep and read it back
    as (freq_hz, amplitude_dbm).
    """
    sva.write(":CALCulate:MARKer1:STATe ON")
    sva.write(":CALCulate:MARKer1:MAXimum")
    freq_hz = float(sva.query(":CALCulate:MARKer1:X?"))
    amp_dbm = float(sva.query(":CALCulate:MARKer1:Y?"))
    return freq_hz, amp_dbm


set_span(
    center_hz=80e6, span_hz=80e6
)  # covers the 60-100 MHz AOD tone band with margin
print("Spectrum analyzer ready.")

Spectrum analyzer ready.


# 3. Simple function generation


In [ ]:
import time

import cupy as cp
import spcm
from spcm import units

MAX_AMPLITUDE_V = 1.0  # conservative; hard ceiling is 2.0 V (see safety notes above)
TEST_FREQUENCY_HZ = (
    10e6  # 10 MHz static tone -- easy to confirm on a scope/spectrum analyzer
)
RUN_SECONDS = 5.0

assert MAX_AMPLITUDE_V <= 2.0, "exceeds hard safety ceiling -- see safety notes above"

with spcm.Card(card_type=spcm.SPCM_TYPE_AO, verbose=True) as card:
    card.card_mode(spcm.SPC_REP_FIFO_SINGLE)
    card.timeout(5 * units.s)

    trigger = spcm.Trigger(card)
    trigger.or_mask(spcm.SPC_TMASK_SOFTWARE)

    channels = spcm.Channels(card, card_enable=spcm.CHANNEL0)
    channels.enable(True)
    channels.output_load(50 * units.ohm)
    channels.amp(MAX_AMPLITUDE_V * units.V)

    clock = spcm.Clock(card)
    clock.mode(spcm.SPC_CM_INTPLL)
    sample_rate_hz = (
        clock.sample_rate(max=True, return_unit=units.Hz).to_base_units().magnitude
    )
    max_value = card.max_sample_value()
    print(f"Sample rate: {sample_rate_hz / 1e6:.1f} MHz")

    notify_samples = 512 * 1024
    dma_buffer_samples = 32 * 1024 * 1024

    scapp_transfer = spcm.SCAPPTransfer(card, direction=spcm.Direction.Generation)
    scapp_transfer.notify_samples(notify_samples)
    scapp_transfer.allocate_buffer(dma_buffer_samples)
    scapp_transfer.start_buffer_transfer(spcm.M2CMD_DATA_STARTDMA)
    cp_dtype = scapp_transfer.numpy_type()

    t_local = cp.arange(notify_samples, dtype=cp.float64) / sample_rate_hz
    phase_s = 0.0
    started = False
    t_start = time.monotonic()
    try:
        for card_buffer in scapp_transfer:
            card_buffer[0, :] = (
                cp.sin(2 * cp.pi * TEST_FREQUENCY_HZ * (t_local + phase_s))
                * 0.4
                * max_value
            ).astype(cp_dtype)
            phase_s += notify_samples / sample_rate_hz

            if not started and scapp_transfer.fill_size_promille() > 800:
                card.start(spcm.M2CMD_CARD_ENABLETRIGGER)
                started = True
                print("Card started -- check the scope/spectrum analyzer now.")

            if time.monotonic() - t_start > RUN_SECONDS:
                break
    finally:
        card.stop(spcm.M2CMD_DATA_STOPDMA | spcm.M2CMD_CARD_STOP)

print(
    f"Done -- static {TEST_FREQUENCY_HZ / 1e6:.0f} MHz tone should have appeared "
    f"on channel 0 for ~{RUN_SECONDS:.0f} s."
)

# 4. Single Ramp testing (observable timescale)


In [ ]:
import spcm

from awg_controller.scripts.atommover_controller import HardwareConfig
from awg_controller.src.awg_control import AODSettings, AWGBatch, RFRamp
from awg_controller.src.scapp import ScappFeeder, ScappFeederConfig

hw = HardwareConfig(
    card_path="/dev/spcm0", max_amplitude_v=1.0
)  # conservative; hard ceiling 2.0 V
aod = AODSettings(
    f_min_v=60e6, f_max_v=100e6, f_min_h=60e6, f_max_h=100e6, grid_rows=1, grid_cols=1
)


def hold_at(freq_hz):
    """Single-tone holding batch at freq_hz, full 40% per-tone budget (only tone on this channel)."""
    return AWGBatch(
        ramps=[
            RFRamp(
                channel=0,
                core=0,
                f_start=freq_hz,
                f_end=freq_hz,
                amplitude_pct=40.0,
                tone_index=0,
            )
        ],
        total_duration_s=0.0,
    )


if "feeder" in globals() and feeder is not None:
    feeder.stop()
    feeder = None
if "card" in globals() and card is not None:
    card.close()
    card = None

card = spcm.Card(hw.card_path).open()
feeder = ScappFeeder(card, hw, aod, ScappFeederConfig(ramp_shape="linear"))
feeder.start(hold_at(aod.f_min_v))
print(
    f"Feeder started -- sample_rate={feeder.sample_rate_hz / 1e6:.1f} MHz, "
    f"holding at {aod.f_min_v / 1e6:.0f} MHz."
)

In [ ]:
OBSERVABLE_RAMP_S = 3.0

ramp_batch = AWGBatch(
    ramps=[
        RFRamp(
            channel=0,
            core=0,
            f_start=aod.f_min_v,
            f_end=aod.f_max_v,
            amplitude_pct=40.0,
            tone_index=0,
        )
    ],
    total_duration_s=OBSERVABLE_RAMP_S,
)
print(
    f"Ramping {aod.f_min_v / 1e6:.0f} -> {aod.f_max_v / 1e6:.0f} MHz over "
    f"{OBSERVABLE_RAMP_S:.0f} s -- watch the spectrum analyzer/scope now."
)
feeder.submit_batch(ramp_batch)  # blocks for ~OBSERVABLE_RAMP_S
feeder.submit_holding(hold_at(aod.f_max_v))
print(
    f"Ramp complete -- now holding at {aod.f_max_v / 1e6:.0f} MHz. "
    f"dropped_transition_count={feeder.dropped_transition_count}"
)

if "sva" in globals() and sva is not None:
    freq_hz, amp_dbm = read_peak()
    print(
        f"SVA1015X peak: {freq_hz / 1e6:.3f} MHz @ {amp_dbm:.1f} dBm "
        f"(expected ~{aod.f_max_v / 1e6:.0f} MHz)"
    )

# 5. Single Ramp testing (experiment timescale)


In [ ]:
from atommovr.utils.timing import MIN_MOVE_DURATION_S

EXPERIMENT_RAMP_S = MIN_MOVE_DURATION_S  # 50 us floor (raised from 1 us) -- this now exceeds this lattice's typical real move range (~3-40 us), so most real moves floor to exactly this value

fast_batch = AWGBatch(
    ramps=[
        RFRamp(
            channel=0,
            core=0,
            f_start=aod.f_max_v,
            f_end=aod.f_min_v,
            amplitude_pct=40.0,
            tone_index=0,
        )
    ],
    total_duration_s=EXPERIMENT_RAMP_S,
)

dropped_before = feeder.dropped_transition_count
feeder.submit_batch(fast_batch)
feeder.submit_holding(hold_at(aod.f_min_v))
dropped_after = feeder.dropped_transition_count

print(
    f"Experiment-timescale ramp: {EXPERIMENT_RAMP_S * 1e6:.1f} us, "
    f"{aod.f_max_v / 1e6:.0f} -> {aod.f_min_v / 1e6:.0f} MHz"
)
print(f"dropped_transition_count: {dropped_before} -> {dropped_after}")
if dropped_after > dropped_before:
    print(
        "WARNING: this transition never reached the DAC -- see ScappFeeder "
        "docstring; consider raising notify_samples."
    )
else:
    print("OK -- transition was rendered within the DMA chunk timing.")

assert feeder.last_error is None, feeder.last_error

if "sva" in globals() and sva is not None:
    freq_hz, amp_dbm = read_peak()
    print(
        f"SVA1015X peak: {freq_hz / 1e6:.3f} MHz @ {amp_dbm:.1f} dBm "
        f"(expected ~{aod.f_min_v / 1e6:.0f} MHz)"
    )

# 6. SVA1015X monitor script (event-triggered recording)


In [ ]:
import subprocess
import sys
import time
from pathlib import Path

# The monitor script (awg_controller/scripts/sva1015x_monitor.py) opens its own
# VISA connection. This instrument is connected over USB-TMC, which only accepts
# one client at a time, so release the notebook's own handle before launching it.
if "sva" in globals() and sva is not None:
    sva.close()
    sva = None
    print("Closed notebook's own SVA1015X connection.")

# Peek the feeder's tone once via a throwaway connection to pick a threshold a
# bit below the actual signal, so this demo reliably triggers a capture burst.
rm = pyvisa.ResourceManager()
probe = rm.open_resource(SVA_RESOURCE)
probe.timeout = 5000
probe.read_termination = "\n"
probe.write_termination = "\n"
probe.write(":CALCulate:MARKer1:STATe ON")
probe.write(":CALCulate:MARKer1:MAXimum")
probe_peak_dbm = float(probe.query(":CALCulate:MARKer1:Y?"))
probe.close()

MONITOR_THRESHOLD_DBM = probe_peak_dbm - 5.0
MONITOR_DURATION_S = 8.0  # None -> stream indefinitely; stop early with the notebook's Interrupt/Stop button
print(
    f"Feeder tone measured at {probe_peak_dbm:.1f} dBm -- monitor threshold set "
    f"to {MONITOR_THRESHOLD_DBM:.1f} dBm."
)

# -u: force the child's stdout/stderr unbuffered so the live-stream cell below
# sees each log line as soon as it's emitted, not batched on process exit.
cmd = [
    sys.executable,
    "-u",
    "awg_controller/scripts/sva1015x_monitor.py",
    "--resource",
    SVA_RESOURCE,
    "--threshold-dbm",
    str(MONITOR_THRESHOLD_DBM),
    "--run-root",
    "runs",
]
if MONITOR_DURATION_S is not None:
    cmd += ["--duration", str(MONITOR_DURATION_S)]

t_launch = time.time()
monitor_proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,  # line-buffered on our end too, for the live-stream cell below
)

# The monitor creates its timestamped run_dir in __init__, before it does
# anything else -- wait (briefly) for that folder to show up so the cells
# below can watch it (events.jsonl / latest_trace.npz) while it's still live.
run_dir = None
for _ in range(50):
    candidates = [
        p for p in Path("runs").glob("sva1015x_*") if p.stat().st_mtime >= t_launch - 1
    ]
    if candidates:
        run_dir = max(candidates, key=lambda p: p.stat().st_mtime)
        break
    time.sleep(0.2)
assert run_dir is not None, "monitor's run directory never appeared -- check it started OK"

print(
    f"Monitor started (pid={monitor_proc.pid}), run_dir={run_dir} -- streaming live "
    f"while the feeder holds at {aod.f_min_v / 1e6:.0f} MHz.\n"
    "Run ONE of the next two cells to watch it live (text log, or a live trace plot) -- "
    "pick one per run, since a blocking cell can't interleave with another live view "
    "in a single kernel. Either way, the final inspect cell works afterward regardless "
    "of which you chose."
)


## 6a. Live trace plot (PyQtGraph, polls `latest_trace.npz`)


In [ ]:
import signal

import numpy as np
import pyqtgraph as pg
from pyqtgraph.Qt import QtCore

# Live plot (sync), native Qt window via pyqtgraph -- far faster redraws than
# a notebook-inline matplotlib clear_output/display loop (no per-frame PNG
# re-encode/re-transfer through the Jupyter comm channel). Still polls
# run_dir/latest_trace.npz -- the file the monitor overwrites atomically every
# sweep -- so no second VISA connection is needed. The achievable update rate
# is capped by the instrument's own sweep time and the SCPI round-trip in
# sva1015x_monitor.py, not by this plot -- see the discussion above; this
# widget just removes the *rendering* bottleneck from the chain.
trace_path = run_dir / "latest_trace.npz"
POLL_MS = 50  # file-check cadence; comfortably under any realistic sweep time

app = pg.mkQApp("SVA1015X live trace")
win = pg.PlotWidget(title="SVA1015X live trace")
win.setLabel("bottom", "Frequency", units="Hz")
win.setLabel("left", "Power", units="dBm")
win.showGrid(x=True, y=True, alpha=0.3)
curve = win.plot([], [], pen=pg.mkPen(width=1.5))
win.resize(900, 450)
win.show()

# Closing the window ends the live view (same as the monitor process exiting)
# without killing the monitor itself -- it keeps running in the background.
win.closeEvent = lambda event: (app.quit(), event.accept())

state = {"last_mtime": None}


def _poll():
    if monitor_proc.poll() is not None:
        timer.stop()
        app.quit()
        return
    if not trace_path.exists():
        return
    mtime = trace_path.stat().st_mtime
    if mtime == state["last_mtime"]:
        return
    try:
        data = np.load(trace_path)
    except (OSError, ValueError):
        return  # caught a write mid-rename (rare, atomic); retry next tick
    state["last_mtime"] = mtime
    curve.setData(data["freqs_hz"], data["trace_dbm"])
    win.setTitle(f"SVA1015X live trace -- {data['timestamp']}")


timer = QtCore.QTimer()
timer.timeout.connect(_poll)
timer.start(POLL_MS)

try:
    app.exec()
except KeyboardInterrupt:
    print("Interrupted from the notebook -- sending SIGINT for a clean shutdown...")
    monitor_proc.send_signal(signal.SIGINT)
    monitor_proc.wait(timeout=15)
finally:
    timer.stop()
    win.close()

print(f"Monitor exited (code {monitor_proc.returncode}).")


## 6b. Live text stream (drains the monitor's stdout)


In [ ]:
import signal

# Live-stream (sync): print each line as the monitor emits it -- its logging
# handler flushes on every record (see JsonlEventHandler/console handler in
# sva1015x_monitor.py), so this cell's output updates in real time, not just
# once at the end. With MONITOR_DURATION_S=None above, interrupt this cell
# (Kernel -> Interrupt, or the Stop button) to end the stream early -- that
# raises KeyboardInterrupt here, which forwards a SIGINT to the monitor so it
# closes any open capture cleanly instead of leaving it dangling.
try:
    for line in monitor_proc.stdout:
        print(line, end="", flush=True)
    monitor_proc.wait()
except KeyboardInterrupt:
    print("\nInterrupted from the notebook -- sending SIGINT for a clean shutdown...")
    monitor_proc.send_signal(signal.SIGINT)
    for line in monitor_proc.stdout:
        print(line, end="", flush=True)
    monitor_proc.wait(timeout=15)

assert monitor_proc.returncode == 0, f"monitor exited with code {monitor_proc.returncode}"
print(f"\nMonitor exited cleanly (code {monitor_proc.returncode}).")


In [ ]:
print(f"Run directory: {run_dir}")

events = (run_dir / "events.jsonl").read_text().strip().splitlines()
print(f"{len(events)} events logged:")
for line in events:
    print(" ", line)

captures = sorted(run_dir.glob("capture_*"))
print(f"{len(captures)} triggered capture burst(s): {[c.name for c in captures]}")
if captures:
    n_traces = len(list(captures[0].glob("trace_*.npz")))
    print(f"  {captures[0].name} contains {n_traces} recorded trace(s) (freqs_hz, trace_dbm, timestamp).")
else:
    print("  No capture triggered -- the feeder's tone may have moved or dropped below threshold.")


# 7. Cleanup


In [ ]:
if "feeder" in globals() and feeder is not None:
    feeder.stop()
    print("Feeder stopped.")
    feeder = None

if "card" in globals() and card is not None:
    card.close()
    print("Card closed.")
    card = None

if "sva" in globals() and sva is not None:
    sva.close()
    print("Spectrum analyzer connection closed.")
    sva = None